In [1]:
from collections.abc import Callable
from functools import partial
from typing import Literal

import numpy as np
import py3Dmol
import seqme as sm
from Bio.Align import PairwiseAligner

/raid/adambiel/venvs/mutation-flows-validation/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _kabsch_transform(ref: np.ndarray, mob: np.ndarray) -> np.ndarray:
    """Rotate and translate mob to minimise RMSD against ref.

    Args:
        ref: Reference coordinates, shape ``(N, 3)``.
        mob: Mobile coordinates to superpose onto ref, shape ``(N, 3)``.

    Returns:
        Superposed coordinates, shape ``(N, 3)``.
    """
    ref_center = ref.mean(axis=0)
    mob_center = mob.mean(axis=0)
    r = ref - ref_center
    m = mob - mob_center

    U, _, Vt = np.linalg.svd(m.T @ r)
    d = np.linalg.det(Vt.T @ U.T)
    D = np.diag([1.0, 1.0, d])
    R = Vt.T @ D @ U.T

    return (m @ R.T) + ref_center


def rmsd(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(((a - b) ** 2).sum() / a.shape[0]))


def indices(s1: str, s2: str) -> np.ndarray:
    res = []
    i = 0
    for c1, c2 in zip(s1, s2, strict=True):
        if c1 != "-":
            if c2 != "-":
                res.append(i)
            i += 1
    return np.array(res, dtype=np.int32)


def compute_rmsd(coords1: np.ndarray, coords2: np.ndarray, seq1: str, seq2: str) -> float:
    align = PairwiseAligner().align(seq1, seq2)[0]
    a_seq1, a_seq2 = align[0], align[1]
    coords1 = coords1[indices(a_seq1, a_seq2)]
    coords2 = coords2[indices(a_seq2, a_seq1)]

    coords2_aligned = _kabsch_transform(coords1, coords2)
    return rmsd(coords1, coords2_aligned)

In [3]:
class RMSD(sm.Metric):
    """Root mean square deviation of atomic positions."""

    def __init__(self, reference: str, folder: Callable[[list[str]], np.ndarray]):
        self.reference = reference
        self.folder = folder

    def __call__(self, sequences: list[str]) -> sm.MetricResult:
        ref_coords = self.folder([self.reference])[0]
        sequences_coords = self.folder(sequences)

        scores = np.array(
            [
                compute_rmsd(seq_coords, ref_coords, seq, self.reference)
                for seq, seq_coords in zip(sequences, sequences_coords, strict=True)
            ]
        )

        return sm.MetricResult(scores.mean().item())

    @property
    def name(self) -> str:
        return "RMSD"

    @property
    def objective(self) -> Literal["minimize", "maximize"]:
        return "minimize"

In [4]:
esm_fold_model = sm.models.ESMFold(cache_dir="/raid/adambiel/models/esmfold", device="cuda:3")

Loading weights: 100%|██████████| 4498/4498 [00:01<00:00, 2400.45it/s]
[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.bias   | MISSING | 
esm.contact_head.regression.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[W814 11:23:26.463016409 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 3 while trying to allocate 104857600 bytes (free: 3801088, total: 11546394624).
[W814 11:23:26.463165723 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 3 while trying to allocate 104857600 bytes (free: 3801088, total: 11546394624).


OutOfMemoryError: CUDA out of memory. Tried to allocate 100.00 MiB. GPU 3 has a total capacity of 10.75 GiB of which 3.62 MiB is free. Including non-PyTorch memory, this process has 10.75 GiB memory in use. Of the allocated memory 10.59 GiB is allocated by PyTorch, and 1.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [5]:
esm_fold_model.device

'cpu'

In [ ]:
cache = sm.Cache(models={"esm-fold": partial(esm_fold_model.fold, convention="atom37", compute_ptm=True)})

In [7]:
esm_fold = cache.model("esm-fold", stack=False)

ptm_fn = lambda sequences: np.array([fold["ptm"] for fold in esm_fold(sequences)])
positions_fn = lambda sequences: [fold["positions"][:, 1, :] for fold in esm_fold(sequences)]  # CA's index = 1
plddt_fn = lambda sequences: np.array([fold["plddt"].mean() for fold in esm_fold(sequences)])
pae_fn = lambda sequences: np.array([fold["pae"].mean() for fold in esm_fold(sequences)])

In [11]:
# Protein folding
atom_indices = [0, 1, 2]  # atoms: N, CA, C
folder = lambda sequences: [fold["positions"][:, atom_indices, :] for fold in esm_fold(sequences)]

In [12]:
sequences = {
    "SAV_STRAV": [
        "MRKIVVAAIAVSLTTVSITASASADPSKDSKAQVSAAEAGITGTWYNQLGSTFIVTAGADGALTGTYESAVGNAESRYVLTGRYDSAPATDGSGTALGWTVAWKNNYRNAHSATTWSGQYVGGAEARINTQWLLTSGTTEANAWKSTLVGHDTFTKVKPSAASIDAAKKAGVNNGNPLDAVQQ"
    ],
    "AVID_CHICK": [
        "MVHATSPLLLLLLLSLALVAPGLSARKCSLTGKWTNDLGSNMTIGAVNSRGEFTGTYITAVTATSNEIKESPLHGTQNTINKRTQPTFGFTVNWKFSESTTVFTGQCFIDRNGKEVLKTMWLLRSSVNDIGDDWKATRVGINIFTRLRTQKE"
    ],
    "GNAT1_HUMAN": [
        "MGAGASAEEKHSRELEKKLKEDAEKDARTVKLLLLGAGESGKSTIVKQMKIIHQDGYSLEECLEFIAIIYGNTLQSILAIVRAMTTLNIQYGDSARQDDARKLMHMADTIEEGTMPKEMSDIIQRLWKDSGIQACFERASEYQLNDSAGYYLSDLERLVTPGYVPTEQDVLRSRVKTTGIIETQFSFKDLNFRMFDVGGQRSERKKWIHCFEGVTCIIFIAALSAYDMVLVEDDEVNRMHESLHLFNSICNHRYFATTSIVLFLNKKDVFFEKIKKAHLSICFPDYDGPNTYEDAGNYIKVQFLELNMRRDVKEIYSHMTCATDTQNVKFVFDAVTDIIIKENLKDCGLF"
    ],
}

sequences["GNAT1_HUMAN (shuffled)"] = sm.utils.shuffle_characters(sequences["GNAT1_HUMAN"])

In [3]:
import torch, triton
print(torch.__version__)
print(torch.version.cuda)
print(triton.__version__)

2.13.0+cu126
12.6
3.7.1


In [2]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(3))

2.13.0+cu126
12.6
True
NVIDIA GeForce RTX 2080 Ti


In [13]:
metrics = [
    RMSD(reference=sequences["SAV_STRAV"][0], folder=positions_fn),
    sm.metrics.ID(predictor=ptm_fn, name="pTM", objective="maximize"),
    sm.metrics.ID(predictor=plddt_fn, name="pLDDT", objective="maximize"),
    sm.metrics.ID(predictor=pae_fn, name="pAE", objective="minimize"),
]

In [14]:
df = sm.evaluate(sequences, metrics)

  0%|          | 0/16 [00:10<?, ?it/s, data=SAV_STRAV, metric=RMSD]


KeyboardInterrupt: 